# Expanded Text Labels and Holdings-Embedding Propagation
## Outcome-free V2 expansion
This notebook converts the audited 300-firm text corpus into directional firm–channel seeds, validates propagation by leaving each seed out in turn, and expands the signals through the same holdings embeddings used in the original pilot. The original 50-firm experiment remains unchanged. Event returns are deliberately not loaded here.


## 1. Environment and immutable inputs
Load only pre-event labels, embeddings, and characteristics.


In [ ]:
from pathlib import Path
import hashlib,importlib,json,sys
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name=='geopolitical_exposure': GEO_ROOT,ROOT=ROOT,ROOT.parent
else: GEO_ROOT=ROOT/'geopolitical_exposure'
if not GEO_ROOT.exists(): raise FileNotFoundError('Open from Asset_Embeddings or geopolitical_exposure.')
sys.path.insert(0,str(GEO_ROOT/'src'))
import embedding_expansion,expanded_label_propagation
importlib.reload(embedding_expansion);importlib.reload(expanded_label_propagation)
from embedding_expansion import l2_normalize,propagate_all_channels,propagate_channel,standardized_characteristics
from expanded_label_propagation import build_directional_seeds,leave_one_seed_out,seed_hash,summarize_leave_one_out
V2_RESULTS=GEO_ROOT/'text_expansion_v2/results';EXPANSION=GEO_ROOT/'embedding_expansion_v2';MANIFESTS=GEO_ROOT/'manifests'
EXPANSION.mkdir(parents=True,exist_ok=True);OUTCOMES_LOADED=False
print('Event outcomes loaded:',OUTCOMES_LOADED)


## 2. Construct and freeze directional firm–channel seeds
Repeated filings for the same firm and channel are collapsed using confidence-weighted signed scores. Ambiguous-direction labels remain in a separate audit file and are excluded from directional propagation.


In [ ]:
final_exposures=pd.read_parquet(V2_RESULTS/'expanded_exposure_panel_final.parquet')
seeds,ambiguous_labels=build_directional_seeds(final_exposures)
seed_sha256=seed_hash(seeds)
seeds.to_parquet(EXPANSION/'directional_firm_channel_seeds.parquet',index=False);ambiguous_labels.to_parquet(EXPANSION/'ambiguous_labels_excluded.parquet',index=False)
seed_summary=pd.DataFrame([{'accepted_document_level_exposures':len(final_exposures),'directional_firm_channel_seeds':len(seeds),'directional_seed_firms':seeds.firm_id.nunique(),'ambiguous_labels_excluded':len(ambiguous_labels),'channels_propagated':seeds.channel.nunique(),'seed_sha256':seed_sha256}])
display(seed_summary);display(seeds.groupby('channel').agg(seeds=('firm_id','size'),firms=('firm_id','nunique'),mean_signed_score=('signed_score','mean')).reset_index())
assert not OUTCOMES_LOADED and not seeds.duplicated(['firm_id','channel']).any()


## 3. Load and align the five pre-event holdings embeddings
Use the identical 2021Q4 representations as the original pilot.


In [ ]:
embedding_paths={'all_s12':ROOT/'active_passive_extension/benchmarks/all_s12_control/2021q4/rs_binary/asset_embeddings_10d.parquet','stable_stock_picker':ROOT/'active_passive_extension/benchmarks/behavioral_embeddings/2021q4/stable_stock_picker/rs_binary/asset_embeddings_10d.parquet','tactical_active':ROOT/'active_passive_extension/benchmarks/behavioral_embeddings/2021q4/tactical_active/rs_binary/asset_embeddings_10d.parquet','benchmark_rebalancer':ROOT/'active_passive_extension/benchmarks/behavioral_embeddings/2021q4/benchmark_rebalancer/rs_binary/asset_embeddings_10d.parquet','passive_like':ROOT/'active_passive_extension/benchmarks/behavioral_embeddings/2021q4/passive_like/rs_binary/asset_embeddings_10d.parquet'}
embedding_frames={};audit=[];reference_firms=None
for name,path in embedding_paths.items():
    frame=pd.read_parquet(path).sort_values('permco').reset_index(drop=True);dimensions=[c for c in frame if c.startswith('embedding_')];firms=tuple(frame.permco.astype(int));reference_firms=firms if reference_firms is None else reference_firms
    embedding_frames[name]=frame;audit.append({'representation':name,'firms':len(frame),'dimensions':len(dimensions),'duplicate_firms':int(frame.permco.duplicated().sum()),'same_ordered_universe':firms==reference_firms,'seed_firms_present':seeds.firm_id.isin(frame.permco).sum()})
embedding_audit=pd.DataFrame(audit);display(embedding_audit)
assert embedding_audit.duplicate_firms.eq(0).all() and embedding_audit.same_ordered_universe.all() and embedding_audit.seed_firms_present.eq(len(seeds)).all()


## 4. Freeze the V2 comparison design
The primary new diagnostic is leave-one-seed-out recovery: each labeled firm is predicted using only the other labeled firms in its channel.


In [ ]:
design={'stage':'expanded_label_embedding_propagation_v2','seed_sha256':seed_sha256,'embedding_date':'2021Q4','similarity':'row_l2_normalized_cosine','propagation':'sum(cosine * signed_score * confidence) / sum(abs(signed_score * confidence))','firm_channel_aggregation':'confidence-weighted mean across accepted documents','ambiguous_direction_policy':'excluded from directional propagation and retained in audit','seed_self_evaluation':False,'leave_one_seed_out':True,'placebo_repetitions':100,'placebo_seed':20220224,'outcomes_loaded':False}
design['sha256']=hashlib.sha256(json.dumps(design,sort_keys=True).encode()).hexdigest();(MANIFESTS/'embedding_expansion_v2_design.json').write_text(json.dumps(design,indent=2))
display(pd.DataFrame([design]))


## 5. Leave-one-seed-out validation
Evaluate whether each representation can recover a labeled firm's signed exposure without allowing that firm to seed its own prediction. This is an outcome-free internal validation, not an event-return test.


In [ ]:
loo_frames=[]
for name,frame in embedding_frames.items():
    dimensions=[c for c in frame if c.startswith('embedding_')];matrix=l2_normalize(frame[dimensions].to_numpy())
    loo_frames.append(leave_one_seed_out(frame,matrix,seeds,name));print(name,'complete')
loo_predictions=pd.concat(loo_frames,ignore_index=True);loo_summary=summarize_leave_one_out(loo_predictions)
loo_predictions.to_parquet(EXPANSION/'leave_one_seed_out_predictions.parquet',index=False);loo_summary.to_parquet(EXPANSION/'leave_one_seed_out_summary.parquet',index=False)
display(loo_summary.sort_values(['channel','spearman_correlation'],ascending=[True,False]))


## 6. Propagate the expanded seeds through all representations
Generate full-universe channel scores after internal validation. Seed firms remain flagged and are excluded from discovery rankings.


In [ ]:
propagation=[]
for name,frame in embedding_frames.items():
    dimensions=[c for c in frame if c.startswith('embedding_')];propagation.append(propagate_all_channels(frame,dimensions,seeds,name));print(name,'complete')
embedding_scores=pd.concat(propagation,ignore_index=True);embedding_scores.to_parquet(EXPANSION/'embedding_propagated_scores.parquet',index=False)
display(embedding_scores.groupby(['representation','channel'],as_index=False).agg(firms=('permco','size'),seeds=('is_seed','sum'),score_min=('propagated_score','min'),score_median=('propagated_score','median'),score_max=('propagated_score','max')))


## 7. Characteristics-similarity benchmark
Propagate the identical seeds through conventional pre-event firm characteristics.


In [ ]:
universe=embedding_frames['all_s12'][['permco','firm_name']].copy();controls=pd.read_parquet(ROOT/'remaining_benchmarks/return_comovement/characteristics/2021q4.parquet')
characteristic_columns=['beta_60m_paper','log_market_equity','log_book_to_market','at_gr1_paper','gp_at_paper','momentum_12_1']
characteristic_frame=universe.merge(controls,on='permco',how='left',validate='one_to_one');characteristic_matrix=standardized_characteristics(characteristic_frame,characteristic_columns)
characteristic_scores=pd.concat([propagate_channel(characteristic_frame,characteristic_matrix,group,'characteristics_similarity') for _,group in seeds.groupby('channel',sort=True)],ignore_index=True)
characteristic_scores.to_parquet(EXPANSION/'characteristic_propagated_scores.parquet',index=False)
characteristic_loo=leave_one_seed_out(characteristic_frame,characteristic_matrix,seeds,'characteristics_similarity');characteristic_loo.to_parquet(EXPANSION/'characteristic_leave_one_out_predictions.parquet',index=False)
all_loo_summary=pd.concat([loo_summary,summarize_leave_one_out(characteristic_loo)],ignore_index=True);all_loo_summary.to_parquet(EXPANSION/'all_leave_one_out_summary.parquet',index=False);display(all_loo_summary.sort_values(['channel','spearman_correlation'],ascending=[True,False]))


## 8. Shuffled-seed placebo
Randomly reassign the channel weights to non-seed firms 100 times to measure accidental structure in the all-S12 embedding.


In [ ]:
rng=np.random.default_rng(20220224);base=embedding_frames['all_s12'];dimensions=[c for c in base if c.startswith('embedding_')];matrix=l2_normalize(base[dimensions].to_numpy());eligible=base.loc[~base.permco.isin(seeds.firm_id),'permco'].to_numpy();placebos=[]
for repetition in range(100):
    for channel,group in seeds.groupby('channel',sort=True):
        pseudo=group.copy();pseudo['firm_id']=rng.choice(eligible,size=len(group),replace=False);scores=propagate_channel(base,matrix,pseudo,'shuffled_embedding_placebo');scores['repetition']=repetition;placebos.append(scores[['repetition','permco','channel','propagated_score']])
placebo_scores=pd.concat(placebos,ignore_index=True);placebo_scores.to_parquet(EXPANSION/'shuffled_embedding_placebos_100.parquet',index=False)
display(pd.DataFrame([{'repetitions':placebo_scores.repetition.nunique(),'rows':len(placebo_scores),'channels':placebo_scores.channel.nunique()}]))


## 9. Compare V1 and V2 expansion rankings
Measure how much the larger label set changes non-seed rankings relative to the original 50-firm pilot.


In [ ]:
v1=pd.read_parquet(GEO_ROOT/'embedding_expansion/embedding_propagated_scores.parquet');v2=embedding_scores.copy();common_channels=sorted(set(v1.channel)&set(v2.channel));rows=[]
for (representation,channel),group in v2[v2.channel.isin(common_channels)].groupby(['representation','channel']):
    old=v1[(v1.representation==representation)&(v1.channel==channel)][['permco','propagated_score']].rename(columns={'propagated_score':'v1_score'});new=group[['permco','propagated_score','is_seed']].rename(columns={'propagated_score':'v2_score'});joined=old.merge(new,on='permco',validate='one_to_one');nonseed=joined[~joined.is_seed]
    rows.append({'representation':representation,'channel':channel,'common_nonseed_firms':len(nonseed),'v1_v2_spearman':nonseed.v1_score.corr(nonseed.v2_score,method='spearman'),'v1_v2_pearson':nonseed.v1_score.corr(nonseed.v2_score)})
v1_v2_comparison=pd.DataFrame(rows);v1_v2_comparison.to_parquet(EXPANSION/'v1_v2_rank_comparison.parquet',index=False);display(v1_v2_comparison)


## 10. Freeze outputs
Hash the V2 artifacts before any event-return evaluation.


In [ ]:
artifact_names=['directional_firm_channel_seeds.parquet','ambiguous_labels_excluded.parquet','leave_one_seed_out_predictions.parquet','all_leave_one_out_summary.parquet','embedding_propagated_scores.parquet','characteristic_propagated_scores.parquet','shuffled_embedding_placebos_100.parquet','v1_v2_rank_comparison.parquet'];inventory=[]
for name in artifact_names:
    path=EXPANSION/name;inventory.append({'artifact':path.stem,'rows':len(pd.read_parquet(path)),'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
inventory=pd.DataFrame(inventory);inventory.to_parquet(EXPANSION/'output_inventory.parquet',index=False)
manifest={'stage':'expanded_label_embedding_propagation_v2','outcomes_loaded':OUTCOMES_LOADED,'seed_sha256':seed_sha256,'design_sha256':design['sha256'],'representations':sorted(embedding_frames),'channels':sorted(seeds.channel.unique()),'common_firms':len(reference_firms),'directional_seed_firms':int(seeds.firm_id.nunique()),'artifacts':inventory.to_dict('records')}
manifest['sha256']=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest();(MANIFESTS/'embedding_expansion_freeze_v2.json').write_text(json.dumps(manifest,indent=2))
display(inventory);print('V2 expansion manifest:',manifest['sha256']);assert not OUTCOMES_LOADED
